# Accumulation Start Offset Slider

Use this notebook to re-accumulate the saved AEDAT4 recording from `runs/camera/laser_selected_cycle_40px_dot12-sync-check` with different `accumulation_start_offset_us` values. It keeps trigger order intact: with five numbers and ten cycles, the grid is 10 rows by 5 columns, not repeat-summed by slot.

In [1]:
from __future__ import annotations

from functools import lru_cache
from math import ceil
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "dmdcontrol").is_dir() and (candidate / "runs").is_dir():
            return candidate
    raise RuntimeError("Could not find repository root from the current notebook directory.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

RUN_DIR = REPO_ROOT / "runs" / "camera" / "laser_selected_cycle_40px_dot12-sync-check"
AEDAT4_PATH = RUN_DIR / "raw.aedat4"
METADATA_PATH = RUN_DIR / "metadata.json"
SUMMARY_PATH = RUN_DIR / "summary.json"

print(f"repo root: {REPO_ROOT}")
print(f"run dir:   {RUN_DIR}")
print(f"aedat4:    {AEDAT4_PATH.exists()} {AEDAT4_PATH}")

repo root: C:\dev\dmdcontrol
run dir:   C:\dev\dmdcontrol\runs\camera\laser_selected_cycle_40px_dot12-sync-check
aedat4:    True C:\dev\dmdcontrol\runs\camera\laser_selected_cycle_40px_dot12-sync-check\raw.aedat4


In [2]:
from dmdcontrol.camera.accumulation import accumulate_events_for_triggers
from dmdcontrol.camera.reprocess_aedat4 import read_aedat4_recording
from dmdcontrol.camera.runs import _events_to_arrays, _process_accumulation_triggers, _trigger_timestamps


def load_json(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


metadata = load_json(METADATA_PATH)
summary = load_json(SUMMARY_PATH)
number_sequence = list(metadata.get("number_sequence") or [])
cycle_length = len(number_sequence) or int(metadata.get("expected_trigger_count") or 5)
default_cycles = int(metadata.get("accumulation_cycles") or 1)
default_window_us = int(
    metadata.get("accumulation_window_us")
    or metadata.get("numbers_exposure_us")
    or summary.get("window_us")
    or 1500
)
default_polarity_mode = str(metadata.get("polarity_mode") or summary.get("polarity_mode") or "signed")
default_offset_us = int(metadata.get("accumulation_start_offset_us") or summary.get("window_start_offset_us") or -250)

print("loading AEDAT4 once; slider redraws reuse this in-memory recording")
recording = read_aedat4_recording(AEDAT4_PATH)
event_arrays = _events_to_arrays(recording.events)
event_timestamps = event_arrays["t"]
width, height = recording.resolution

print(f"resolution: {width} x {height}")
print(f"event batches: {len(recording.events):,}; events: {recording.stats['event_count']:,}")
print(f"triggers: {recording.stats['trigger_count']:,}; trigger edges: {recording.stats['trigger_edges']}")
print(f"cycle length: {cycle_length}; default cycles: {default_cycles}; default window: {default_window_us} us")

loading AEDAT4 once; slider redraws reuse this in-memory recording
resolution: 320 x 240
event batches: 756; events: 528,257
triggers: 5,676; trigger edges: {'rising': 2838, 'triggertype.external_signal_pulse': 2838}
cycle length: 5; default cycles: 10; default window: 1500 us


In [3]:
def _stages_for(offset_us: int, cycles: int, window_us: int):
    return _process_accumulation_triggers(
        recording.triggers,
        event_timestamps,
        window_us=int(window_us),
        window_start_offset_us=int(offset_us),
        max_accumulation_triggers=None,
        trigger_cycle_length=cycle_length,
        accumulation_cycles=int(cycles),
    )


@lru_cache(maxsize=16)
def accumulate_for_offset(offset_us: int, cycles: int, window_us: int, polarity_mode: str):
    stages = _stages_for(offset_us, cycles, window_us)
    frames = accumulate_events_for_triggers(
        recording.events,
        stages.final,
        resolution=recording.resolution,
        window_us=int(window_us),
        polarity_mode=polarity_mode,
        window_start_offset_us=int(offset_us),
    )
    trigger_ts = _trigger_timestamps(stages.final)
    return frames, trigger_ts, stages.alignment_metadata, stages.cycle_limit_metadata


def display_values(frames: np.ndarray, polarity_view: str, tone_curve: str, gamma: float) -> tuple[np.ndarray, str]:
    frames = frames.astype(np.float32, copy=False)
    if polarity_view == "signed":
        return frames, "signed"
    if polarity_view == "positive":
        values = np.maximum(frames, 0)
    elif polarity_view == "negative":
        values = np.maximum(-frames, 0)
    else:
        values = np.abs(frames)

    if tone_curve == "log":
        values = np.log1p(values)
    elif tone_curve == "gamma":
        values = np.power(values, float(gamma))
    return values, "gray"


def robust_limit(values: np.ndarray, percentile: float, signed: bool) -> float:
    source = np.abs(values) if signed else values
    nonzero = source[source > 0]
    if nonzero.size == 0:
        return 1.0
    return max(float(np.percentile(nonzero, percentile)), 1e-6)


def render_offset(
    offset_us: int = default_offset_us,
    cycles: int = default_cycles,
    window_us: int = default_window_us,
    polarity_mode: str = default_polarity_mode,
    polarity_view: str = "magnitude",
    tone_curve: str = "log",
    gamma: float = 0.5,
    vmax_percentile: float = 99.5,
    flip_x: bool = True,
):
    frames, trigger_ts, alignment, cycle_info = accumulate_for_offset(
        int(offset_us), int(cycles), int(window_us), str(polarity_mode)
    )
    if flip_x:
        frames = frames[:, :, ::-1]

    values, color_kind = display_values(frames, polarity_view, tone_curve, gamma)
    signed = color_kind == "signed"
    limit = robust_limit(values, vmax_percentile, signed=signed)
    cmap = "coolwarm" if signed else "gray"
    vmin, vmax = (-limit, limit) if signed else (0, limit)

    frame_count = values.shape[0]
    cols = max(1, int(cycle_length))
    rows = max(1, ceil(frame_count / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(2.35 * cols, 2.35 * rows), squeeze=False)

    first_trigger = int(trigger_ts[0]) if len(trigger_ts) else 0
    counts = np.count_nonzero(np.abs(frames) > 0, axis=(1, 2)) if frame_count else np.array([])

    for index, ax in enumerate(axes.ravel()):
        ax.axis("off")
        if index >= frame_count:
            continue
        ax.imshow(values[index], cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
        slot = index % cols
        cycle = index // cols + 1
        label = number_sequence[slot] if slot < len(number_sequence) else slot + 1
        dt = int(trigger_ts[index] - first_trigger) if len(trigger_ts) else 0
        ax.set_title(f"{index + 1}: {label}  c{cycle}\n+{dt} us", fontsize=8)

    plt.suptitle(
        f"offset {int(offset_us)} us | window {int(window_us)} us | "
        f"{frame_count} ordered frames | view={polarity_view}/{tone_curve}",
        y=1.0,
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()

    print(
        "raw rising:", alignment.get("input_trigger_count"),
        "aligned:", alignment.get("aligned_trigger_count"),
        "selected:", cycle_info.get("selected_trigger_count"),
        "cycles:", cycle_info.get("selected_cycle_indices"),
    )
    if counts.size:
        print("nonzero pixels per frame: min", int(counts.min()), "median", int(np.median(counts)), "max", int(counts.max()))
    return frames

## Interactive Offset Sweep

Move the offset slider to shift each accumulation window relative to its trigger. Negative values start before the trigger. The redraw keeps frames in trigger order, so frame 1 through frame 50 should remain the actual acquisition sequence.

In [4]:
try:
    import ipywidgets as widgets
    HAVE_WIDGETS = True
except ModuleNotFoundError:
    HAVE_WIDGETS = False
    print("ipywidgets is not installed in this kernel. For the slider, run `%pip install ipywidgets`, restart the kernel, then rerun this notebook.")


if HAVE_WIDGETS:
    available_full_cycles = max(1, recording.stats["trigger_edges"].get("rising", 0) // max(1, cycle_length))
    offset_slider = widgets.IntSlider(
        value=default_offset_us,
        min=-1500,
        max=750,
        step=25,
        description="offset us",
        continuous_update=False,
        layout=widgets.Layout(width="520px"),
    )
    cycles_slider = widgets.IntSlider(
        value=min(default_cycles, available_full_cycles),
        min=1,
        max=max(1, min(25, available_full_cycles)),
        step=1,
        description="cycles",
        continuous_update=False,
        layout=widgets.Layout(width="520px"),
    )
    window_slider = widgets.IntSlider(
        value=default_window_us,
        min=100,
        max=max(default_window_us * 2, 2000),
        step=25,
        description="window us",
        continuous_update=False,
        layout=widgets.Layout(width="520px"),
    )
    polarity_mode_dropdown = widgets.Dropdown(
        value=default_polarity_mode if default_polarity_mode in {"positive", "signed", "ignore"} else "signed",
        options=["signed", "positive", "ignore"],
        description="accumulate",
    )
    polarity_view_dropdown = widgets.Dropdown(
        value="magnitude",
        options=["magnitude", "positive", "negative", "signed"],
        description="view",
    )
    tone_dropdown = widgets.Dropdown(value="log", options=["log", "gamma", "linear"], description="tone")
    gamma_slider = widgets.FloatSlider(
        value=0.5,
        min=0.2,
        max=1.5,
        step=0.05,
        description="gamma",
        continuous_update=False,
        layout=widgets.Layout(width="520px"),
    )
    percentile_slider = widgets.FloatSlider(
        value=99.5,
        min=90.0,
        max=100.0,
        step=0.1,
        description="vmax %",
        continuous_update=False,
        layout=widgets.Layout(width="520px"),
    )
    flip_checkbox = widgets.Checkbox(value=True, description="flip x for viewing")
    out = widgets.Output()

    def redraw(_=None):
        with out:
            clear_output(wait=True)
            render_offset(
                offset_us=offset_slider.value,
                cycles=cycles_slider.value,
                window_us=window_slider.value,
                polarity_mode=polarity_mode_dropdown.value,
                polarity_view=polarity_view_dropdown.value,
                tone_curve=tone_dropdown.value,
                gamma=gamma_slider.value,
                vmax_percentile=percentile_slider.value,
                flip_x=flip_checkbox.value,
            )

    for widget in [
        offset_slider,
        cycles_slider,
        window_slider,
        polarity_mode_dropdown,
        polarity_view_dropdown,
        tone_dropdown,
        gamma_slider,
        percentile_slider,
        flip_checkbox,
    ]:
        widget.observe(redraw, names="value")

    display(widgets.VBox([
        offset_slider,
        cycles_slider,
        window_slider,
        widgets.HBox([polarity_mode_dropdown, polarity_view_dropdown, tone_dropdown]),
        gamma_slider,
        percentile_slider,
        flip_checkbox,
    ]))
    display(out)
    redraw()
else:
    MANUAL_OFFSET_US = default_offset_us
    frames = render_offset(offset_us=MANUAL_OFFSET_US, cycles=default_cycles, window_us=default_window_us)

Output()

## Manual Calls

If you want to compare exact offsets without the widget, run calls like these. Each call reuses the already loaded AEDAT4 object.

In [5]:
# Example manual checks:
# frames_m250 = render_offset(offset_us=-250, cycles=10, window_us=1500)
# frames_0 = render_offset(offset_us=0, cycles=10, window_us=1500)
# frames_p250 = render_offset(offset_us=250, cycles=10, window_us=1500)

# The returned array is shaped: (ordered_trigger_frames, height, width).
# Nothing is written back to the run directory unless you explicitly add a save call.